# EEG Abnormality Prediction

Template-driven healthcare machine learning notebook for the HCTM EEG Abnormality project. This notebook is research-only and is not intended for clinical deployment, diagnosis, or treatment decisions.

# 1. Data Loading

This section defines the EEG project context, imports libraries, loads the HCTM dataset, and constructs the binary EEG abnormality target.

## 1.1 Project Setup

In [1]:
# Core libraries
import os
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
SHAP_OUTPUT_DIR = OUTPUT_DIR / "shap"
MPLCONFIG_DIR = OUTPUT_DIR / "matplotlib"
for directory in [OUTPUT_DIR, SHAP_OUTPUT_DIR, MPLCONFIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.environ["MPLCONFIGDIR"] = str(MPLCONFIG_DIR)
os.environ["PYTHONHASHSEED"] = "42"
warnings.filterwarnings("ignore")

import numpy as np
if not hasattr(np, "NaN"):
    np.NaN = np.nan
import pandas as pd
import scipy.stats as scipy_stats

if not hasattr(scipy_stats, "binom_test"):
    scipy_stats.binom_test = lambda x, n=None, p=0.5, alternative="two-sided": scipy_stats.binomtest(
        int(round(x)), n=int(n), p=p, alternative=alternative
    ).pvalue

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid, StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

from boruta import BorutaPy
from BorutaShap import BorutaShap
from catboost import CatBoostClassifier, Pool
import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Random seed: {RANDOM_STATE}")

Project root: Z:\R&D\EEG
Output directory: Z:\R&D\EEG\outputs
Random seed: 42


## 1.2 Project Configuration

Project-specific constants are defined here. The remaining notebook follows the healthcare ML template workflow.

In [2]:
PROJECT_NAME = "EEG Abnormality Prediction"
SHORT_NAME = "EEG Abnormality"
DATA_PATH = Path("Dataset/EEG ABNORMALITY DATASET.xlsx")
SHEET_NAME = "Raw Data Set"
TARGET_COLUMN = "EEG FINDINGS"
TARGET_BINARY = "EEG_ABNORMAL"
POSITIVE_CLASS_LABEL = "Abnormal EEG"
TEST_SIZE = 0.20
CV_FOLDS = 5
BORUTASHAP_N_TRIALS = 100
BORUTASHAP_PERCENTILE = 100
BORUTASHAP_PVALUE = 0.05
BORUTASHAP_CATBOOST_ITERATIONS = 80

DATASET_SOURCE = "Hospital Canselor Tuanku Muhriz (HCTM)"
RESEARCH_ONLY_NOTE = (
    "Research-only retrospective single-centre analysis. "
    "No clinical deployment, diagnosis, or treatment decision claims are made."
)

config_summary = pd.DataFrame(
    {
        "item": [
            "project",
            "dataset_source",
            "dataset_path",
            "sheet_name",
            "target_column",
            "binary_target",
            "positive_class",
            "test_size",
            "cv_folds",
        ],
        "value": [
            PROJECT_NAME,
            DATASET_SOURCE,
            str(DATA_PATH),
            SHEET_NAME,
            TARGET_COLUMN,
            TARGET_BINARY,
            POSITIVE_CLASS_LABEL,
            TEST_SIZE,
            CV_FOLDS,
        ],
    }
)
display(config_summary)

,item,value
0,project,EEG Abnormality Prediction
1,dataset_source,Hospital Canselor Tuanku Muhriz (HCTM)
2,dataset_path,Dataset\EEG ABNORMALITY DATASET.xlsx
3,sheet_name,Raw Data Set
4,target_column,EEG FINDINGS
5,binary_target,EEG_ABNORMAL
6,positive_class,Abnormal EEG
7,test_size,0.2
8,cv_folds,5


## 1.3 Load Dataset

In [3]:
def load_dataset(path, sheet_name=None):
    suffix = path.suffix.lower()
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path, sheet_name=sheet_name)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported dataset suffix: {suffix}")


if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

raw_data = load_dataset(DATA_PATH, SHEET_NAME)
raw_shape = raw_data.shape
raw_columns_original = list(raw_data.columns)
raw_data = raw_data.copy()
raw_data.columns = raw_data.columns.astype(str).str.strip()
raw_columns_cleaned = list(raw_data.columns)

column_check = pd.DataFrame(
    {
        "column": raw_columns_cleaned,
        "dtype": [str(raw_data[col].dtype) for col in raw_columns_cleaned],
        "missing_n": [int(raw_data[col].isna().sum()) for col in raw_columns_cleaned],
        "missing_pct": [float(raw_data[col].isna().mean() * 100) for col in raw_columns_cleaned],
    }
)

changed_columns = [before for before, after in zip(raw_columns_original, raw_columns_cleaned) if before != after]
print(f"Raw dataset shape: {raw_shape[0]} rows x {raw_shape[1]} columns")
print("Column-name whitespace was stripped in memory only; the raw workbook was not edited.")
print(f"Columns changed after stripping whitespace: {changed_columns}")
display(column_check)

Raw dataset shape: 288 rows x 50 columns
Column-name whitespace was stripped in memory only; the raw workbook was not edited.
Columns changed after stripping whitespace: ['CREATININE ']


,column,dtype,missing_n,missing_pct
0,AGE,int64,0,0.000000
1,GENDER,str,0,0.000000
2,RACE,str,0,0.000000
3,SMOKING,str,0,0.000000
4,ALCOHOL,str,0,0.000000
5,HYPERTENSION,str,0,0.000000
6,DIABETES MELLITUS,str,0,0.000000
7,HYPERLIPIDEMIA,str,0,0.000000
8,STROKE,str,0,0.000000
9,TB,str,0,0.000000


## 1.4 Target Definition

The target maps `Normal` EEG findings to 0 and all clinically non-normal EEG findings to 1. Rows with missing or invalid target values are removed before mapping.

In [4]:
data = raw_data.copy()

if TARGET_COLUMN not in data.columns:
    raise KeyError(f"Required target column not found: {TARGET_COLUMN}")


def clean_text_value(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text in {"", "-", "NA", "N/A", "None", "NULL", "nan"}:
        return np.nan
    return text


target_clean = data[TARGET_COLUMN].map(clean_text_value)
valid_target_mask = target_clean.notna()
invalid_target_rows = int((~valid_target_mask).sum())
if invalid_target_rows:
    print(f"Dropping {invalid_target_rows} rows with missing/invalid target values.")

data = data.loc[valid_target_mask].copy()
target_clean = target_clean.loc[valid_target_mask]
data[TARGET_BINARY] = np.where(target_clean.str.lower().eq("normal"), 0, 1).astype(int)

raw_target_distribution = target_clean.value_counts(dropna=False).rename_axis("raw_eeg_finding").reset_index(name="n")
binary_target_distribution = (
    data[TARGET_BINARY]
    .map({0: "Normal EEG (0)", 1: "Abnormal EEG (1)"})
    .value_counts()
    .rename_axis("target_class")
    .reset_index(name="n")
)
binary_target_distribution["pct"] = binary_target_distribution["n"] / binary_target_distribution["n"].sum() * 100

print(f"Analysis dataset shape after target cleaning: {data.shape[0]} rows x {data.shape[1]} columns")
display(raw_target_distribution)
display(binary_target_distribution)

Analysis dataset shape after target cleaning: 288 rows x 51 columns


,raw_eeg_finding,n
0,Generalised theta or delta,158
1,Normal,77
2,Focal slowing,31
3,Epileptic,22


,target_class,n,pct
0,Abnormal EEG (1),211,73.263889
1,Normal EEG (0),77,26.736111


# 2. Preprocessing

This section implements leakage-aware feature exclusion, EDA-driven cleaning, stratified splitting, train-fitted preprocessing, and model-specific feature selection before benchmarking.

## 2.1 Feature Exclusion

The following exclusions remove leakage, downstream outcomes, post-hoc interpretations, unavailable fields, high-missing free text, and redundant raw fields. This is feature exclusion, not feature selection.

In [5]:
LEAKAGE_COLUMNS = [
    "CAUSES",
    "LENGTH OF HOSPITALISATION",
    "OUTCOME OF HOSPITALISATION",
    "INDICATION FOR EEG",
    "DIAGNOSIS",
    "DXT/FBS/RBS/HbA1c",
    "OTHERS",
    "TOTAL CHOLESTEROL",
    "BLOOD PRESSURE ON PRESENTATION",
]

exclusion_reasons = {
    "CAUSES": "Leakage / diagnostic pathway / possible post-hoc clinical interpretation",
    "LENGTH OF HOSPITALISATION": "Post-outcome / downstream variable",
    "OUTCOME OF HOSPITALISATION": "Post-outcome",
    "INDICATION FOR EEG": "EEG-ordering pathway / potential target proxy",
    "DIAGNOSIS": "Target proxy / post-clinical interpretation",
    "DXT/FBS/RBS/HbA1c": "Excluded raw/related glucose measurement field",
    "OTHERS": "99 percent missing / free-text / low utility",
    "TOTAL CHOLESTEROL": "Excluded based on prior redundancy/project decision",
    "BLOOD PRESSURE ON PRESENTATION": "Raw combined BP field; separated BP fields are used if retained",
}

present_leakage_columns = [col for col in LEAKAGE_COLUMNS if col in data.columns]
missing_leakage_columns = [col for col in LEAKAGE_COLUMNS if col not in data.columns]

exclusion_summary = pd.DataFrame(
    {
        "column": LEAKAGE_COLUMNS,
        "present_in_dataset": [col in data.columns for col in LEAKAGE_COLUMNS],
        "reason": [exclusion_reasons[col] for col in LEAKAGE_COLUMNS],
    }
)

display(exclusion_summary)
print(f"Excluded leakage/downstream/unavailable columns present: {len(present_leakage_columns)}")
print(f"Specified exclusion columns not present: {missing_leakage_columns}")


,column,present_in_dataset,reason
0,CAUSES,True,Leakage / diagnostic pathway / possible post-h...
1,LENGTH OF HOSPITALISATION,True,Post-outcome / downstream variable
2,OUTCOME OF HOSPITALISATION,True,Post-outcome
3,INDICATION FOR EEG,True,EEG-ordering pathway / potential target proxy
4,DIAGNOSIS,True,Target proxy / post-clinical interpretation
5,DXT/FBS/RBS/HbA1c,True,Excluded raw/related glucose measurement field
6,OTHERS,True,99 percent missing / free-text / low utility
7,TOTAL CHOLESTEROL,True,Excluded based on prior redundancy/project dec...
8,BLOOD PRESSURE ON PRESENTATION,True,Raw combined BP field; separated BP fields are...


Excluded leakage/downstream/unavailable columns present: 9
Specified exclusion columns not present: []


## 2.2 Feature Type Split/Definition

Feature groups are defined explicitly so demographic categories, binary clinical status variables, laboratory values, and CatBoost categorical columns are handled according to the project context.

In [6]:
numeric_columns_spec = [
    "AGE",
    "Systolic",
    "Diastolic",
    "HAEMOGLOBIN",
    "PLATELET COUNT",
    "WHITE CELL COUNT",
    "BLOOD UREA",
    "POTASSIUM",
    "CREATININE",
    "SODIUM",
    "TOTAL PROTEIN",
    "ALBUMIN",
    "ALANINE TRANSAMINASE",
    "CALCIUM",
    "TRIGLYCERIDE(TG)",
    "LDL",
    "HDL",
]

binary_clinical_categorical_spec = [
    "SMOKING",
    "ALCOHOL",
    "HYPERTENSION",
    "DIABETES MELLITUS",
    "HYPERLIPIDEMIA",
    "STROKE",
    "TB",
    "AF",
    "IHD",
    "CONGESTIVE CARDIAC FAILURE",
    "COPD",
    "BRONCHIAL ASTHMA",
    "CKD",
    "PSYCHIATRIC DISORDER",
    "BRAIN TUMOUR",
    "HIV",
    "SEIZURES",
    "FOCAL SEIZURE",
    "GENERALISED SEIZURE",
]

demographic_categorical_spec = ["GENDER", "RACE"]
nonbinary_clinical_categorical_spec = ["Blood Sugar Control", "RADIOLOGICAL IMAGING"]

numeric_columns = [col for col in numeric_columns_spec if col in data.columns]
binary_clinical_categorical_columns = [col for col in binary_clinical_categorical_spec if col in data.columns]
demographic_categorical_columns = [col for col in demographic_categorical_spec if col in data.columns]
nonbinary_clinical_categorical_columns = [col for col in nonbinary_clinical_categorical_spec if col in data.columns]
categorical_columns = (
    binary_clinical_categorical_columns
    + demographic_categorical_columns
    + nonbinary_clinical_categorical_columns
)

model_data = data.drop(columns=present_leakage_columns, errors="ignore").copy()
feature_columns = [
    col
    for col in numeric_columns + categorical_columns
    if col in model_data.columns and col not in {TARGET_COLUMN, TARGET_BINARY}
]

missing_expected = sorted(
    set(numeric_columns_spec + binary_clinical_categorical_spec + demographic_categorical_spec + nonbinary_clinical_categorical_spec)
    - set(feature_columns)
    - set(LEAKAGE_COLUMNS)
)

feature_type_summary = pd.DataFrame(
    [
        {"feature_group": "numerical", "n": len(numeric_columns), "features": numeric_columns},
        {
            "feature_group": "binary_clinical_categorical",
            "n": len(binary_clinical_categorical_columns),
            "features": binary_clinical_categorical_columns,
        },
        {"feature_group": "demographic_categorical", "n": len(demographic_categorical_columns), "features": demographic_categorical_columns},
        {
            "feature_group": "nonbinary_clinical_categorical",
            "n": len(nonbinary_clinical_categorical_columns),
            "features": nonbinary_clinical_categorical_columns,
        },
    ]
)

print(f"Retained modeling features before feature selection: {len(feature_columns)}")
print(f"Missing expected retained features after in-memory column cleanup: {missing_expected}")
display(feature_type_summary)


Retained modeling features before feature selection: 40
Missing expected retained features after in-memory column cleanup: []


,feature_group,n,features
0,numerical,17,"[AGE, Systolic, Diastolic, HAEMOGLOBIN, PLATEL..."
1,binary_clinical_categorical,19,"[SMOKING, ALCOHOL, HYPERTENSION, DIABETES MELL..."
2,demographic_categorical,2,"[GENDER, RACE]"
3,nonbinary_clinical_categorical,2,"[Blood Sugar Control, RADIOLOGICAL IMAGING]"


## 2.3 EDA-Driven Data Cleaning

The missingness and outlier policies below are implemented using training-set fitted values only. Medical outliers are not deleted blindly.

In [7]:
numerical_missingness_report = pd.DataFrame(
    [
        ("HAEMOGLOBIN", 3, 1.041667),
        ("PLATELET COUNT", 3, 1.041667),
        ("WHITE CELL COUNT", 3, 1.041667),
        ("BLOOD UREA", 4, 1.388889),
        ("POTASSIUM", 2, 0.694444),
        ("CREATININE", 2, 0.694444),
        ("SODIUM", 2, 0.694444),
        ("TOTAL PROTEIN", 5, 1.736111),
        ("ALBUMIN", 3, 1.041667),
        ("ALANINE TRANSAMINASE", 4, 1.388889),
        ("CALCIUM", 4, 1.388889),
        ("TRIGLYCERIDE(TG)", 52, 18.055556),
        ("LDL", 51, 17.708333),
        ("HDL", 52, 18.055556),
    ],
    columns=["feature", "missing_values", "missing_pct"],
)

categorical_missingness_report = pd.DataFrame(
    [
        ("RADIOLOGICAL IMAGING", 13, 4.513889, "Candidate predictor; preserve imaging findings / Not performed category"),
        ("Blood Sugar Control", 42, 14.583333, "Missing/not tested/not documented as 2"),
        ("DXT/FBS/RBS/HbA1c", 40, 13.888889, "Excluded"),
        ("OTHERS", 286, 99.305556, "Excluded"),
    ],
    columns=["feature", "missing_values", "missing_pct", "handling"],
)

outlier_policy = pd.DataFrame(
    [
        ("AGE", "Retain clinically plausible values"),
        ("Systolic", "Retain clinically plausible values"),
        ("Diastolic", "Retain clinically plausible values"),
        ("PLATELET COUNT", "Train-derived tail capping and log transform if skewed"),
        ("WHITE CELL COUNT", "Train-derived tail capping and log transform if skewed"),
        ("BLOOD UREA", "Train-derived tail capping and log transform if skewed"),
        ("CREATININE", "Train-derived tail capping and log transform if skewed"),
        ("ALANINE TRANSAMINASE", "Train-derived tail capping and log transform if skewed"),
        ("TRIGLYCERIDE(TG)", "Missingness flag, median imputation, log transform if skewed"),
        ("LDL", "Missingness flag and retain/cap only if invalid"),
        ("HDL", "Missingness flag and retain/cap only if invalid"),
    ],
    columns=["feature", "handling"],
)

missingness_policy = pd.DataFrame(
    [
        ("Low-missing numerical features", "Training-set median"),
        ("TRIGLYCERIDE(TG), LDL, HDL", "Training-set median plus missingness flag"),
        ("Binary clinical categorical features", "Missing/unmapped as 2 where clinically appropriate"),
        ("Demographic categorical features", "Training-set mode imputation if missing"),
        ("Blood Sugar Control", "Missing/unknown/not tested as 2"),
        ("RADIOLOGICAL IMAGING", "Candidate predictor; preserve imaging findings and missing/not performed category"),
    ],
    columns=["feature_or_group", "handling_rule"],
)

display(numerical_missingness_report)
display(categorical_missingness_report)
display(outlier_policy)
display(missingness_policy)


,feature,missing_values,missing_pct
0,HAEMOGLOBIN,3,1.041667
1,PLATELET COUNT,3,1.041667
2,WHITE CELL COUNT,3,1.041667
3,BLOOD UREA,4,1.388889
4,POTASSIUM,2,0.694444
5,CREATININE,2,0.694444
6,SODIUM,2,0.694444
7,TOTAL PROTEIN,5,1.736111
8,ALBUMIN,3,1.041667
9,ALANINE TRANSAMINASE,4,1.388889


,feature,missing_values,missing_pct,handling
0,RADIOLOGICAL IMAGING,13,4.513889,Candidate predictor; preserve imaging findings...
1,Blood Sugar Control,42,14.583333,Missing/not tested/not documented as 2
2,DXT/FBS/RBS/HbA1c,40,13.888889,Excluded
3,OTHERS,286,99.305556,Excluded


,feature,handling
0,AGE,Retain clinically plausible values
1,Systolic,Retain clinically plausible values
2,Diastolic,Retain clinically plausible values
3,PLATELET COUNT,Train-derived tail capping and log transform i...
4,WHITE CELL COUNT,Train-derived tail capping and log transform i...
5,BLOOD UREA,Train-derived tail capping and log transform i...
6,CREATININE,Train-derived tail capping and log transform i...
7,ALANINE TRANSAMINASE,Train-derived tail capping and log transform i...
8,TRIGLYCERIDE(TG),"Missingness flag, median imputation, log trans..."
9,LDL,Missingness flag and retain/cap only if invalid


,feature_or_group,handling_rule
0,Low-missing numerical features,Training-set median
1,"TRIGLYCERIDE(TG), LDL, HDL",Training-set median plus missingness flag
2,Binary clinical categorical features,Missing/unmapped as 2 where clinically appropr...
3,Demographic categorical features,Training-set mode imputation if missing
4,Blood Sugar Control,Missing/unknown/not tested as 2
5,RADIOLOGICAL IMAGING,Candidate predictor; preserve imaging findings...


In [8]:
MISSING_TOKENS = {"", " ", "-", "NA", "N/A", "na", "n/a", "None", "none", "NULL", "null", "nan", "NaN"}
YES_TOKENS = {"yes", "y", "true", "1", "present", "positive"}
NO_TOKENS = {"no", "n", "false", "0", "absent", "negative"}
UNKNOWN_TOKENS = {"unknown", "not known", "not documented", "not done", "not performed", "not tested", "not applicable", "missing"}

LIPID_MISSING_FLAG_FEATURES = ["TRIGLYCERIDE(TG)", "LDL", "HDL"]
CAPPING_CANDIDATES = ["PLATELET COUNT", "WHITE CELL COUNT", "BLOOD UREA", "CREATININE", "ALANINE TRANSAMINASE"]
LOG_TRANSFORM_CANDIDATES = CAPPING_CANDIDATES + ["TRIGLYCERIDE(TG)"]


def normalize_missing_tokens(df):
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].map(lambda value: np.nan if pd.isna(value) or str(value).strip() in MISSING_TOKENS else str(value).strip())
    return df


def coerce_numeric(series):
    cleaned = series.map(lambda value: np.nan if pd.isna(value) or str(value).strip() in MISSING_TOKENS else value)
    cleaned = cleaned.astype("string").str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce").astype(float)


def map_binary_clinical_value(value):
    if pd.isna(value):
        return "2"
    text = str(value).strip().lower()
    if text in MISSING_TOKENS or text in UNKNOWN_TOKENS:
        return "2"
    if text in YES_TOKENS:
        return "1"
    if text in NO_TOKENS:
        return "0"
    return "2"


def clean_category_value(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text in MISSING_TOKENS or text.lower() in UNKNOWN_TOKENS:
        return np.nan
    return text


class ClinicalPreprocessor:
    def __init__(
        self,
        numeric_columns,
        binary_categorical_columns,
        demographic_categorical_columns,
        nonbinary_categorical_columns,
        missing_flag_features,
        capping_candidates,
        log_transform_candidates,
    ):
        self.numeric_columns = list(numeric_columns)
        self.binary_categorical_columns = list(binary_categorical_columns)
        self.demographic_categorical_columns = list(demographic_categorical_columns)
        self.nonbinary_categorical_columns = list(nonbinary_categorical_columns)
        self.missing_flag_features = [col for col in missing_flag_features if col in self.numeric_columns]
        self.missing_flag_columns = [f"{col}__missing_flag" for col in self.missing_flag_features]
        self.capping_candidates = [col for col in capping_candidates if col in self.numeric_columns]
        self.log_transform_candidates = [col for col in log_transform_candidates if col in self.numeric_columns]
        self.numeric_medians_ = {}
        self.categorical_modes_ = {}
        self.cap_bounds_ = {}
        self.log_transform_columns_ = []

    def fit(self, X, y=None):
        X_norm = normalize_missing_tokens(X)
        numeric_train = pd.DataFrame(index=X_norm.index)
        for col in self.numeric_columns:
            numeric_train[col] = coerce_numeric(X_norm[col])
            median = numeric_train[col].median()
            self.numeric_medians_[col] = 0.0 if pd.isna(median) else float(median)
            if col in self.capping_candidates:
                observed = numeric_train[col].dropna()
                lower = self.numeric_medians_[col] if observed.empty else float(observed.quantile(0.01))
                upper = self.numeric_medians_[col] if observed.empty else float(observed.quantile(0.99))
                if lower > upper:
                    lower, upper = upper, lower
                self.cap_bounds_[col] = (lower, upper)

        for col in self.demographic_categorical_columns:
            cleaned = X_norm[col].map(clean_category_value)
            mode_values = cleaned.dropna().mode()
            self.categorical_modes_[col] = "Unknown" if mode_values.empty else str(mode_values.iloc[0])

        numeric_for_skew = numeric_train.copy()
        for col in self.numeric_columns:
            numeric_for_skew[col] = numeric_for_skew[col].fillna(self.numeric_medians_[col])
            if col in self.cap_bounds_:
                lower, upper = self.cap_bounds_[col]
                numeric_for_skew[col] = numeric_for_skew[col].clip(lower=lower, upper=upper)

        self.log_transform_columns_ = [
            col
            for col in self.log_transform_candidates
            if numeric_for_skew[col].min() >= 0 and abs(numeric_for_skew[col].skew()) >= 1.0
        ]
        self.numeric_model_columns_ = self.numeric_columns + self.missing_flag_columns
        self.categorical_model_columns_ = (
            self.binary_categorical_columns
            + self.demographic_categorical_columns
            + self.nonbinary_categorical_columns
        )
        self.feature_order_ = self.numeric_model_columns_ + self.categorical_model_columns_
        return self

    def transform(self, X):
        X_norm = normalize_missing_tokens(X)
        transformed = pd.DataFrame(index=X_norm.index)
        for col in self.numeric_columns:
            numeric = coerce_numeric(X_norm[col])
            if col in self.missing_flag_features:
                transformed[f"{col}__missing_flag"] = numeric.isna().astype(int)
            numeric = numeric.fillna(self.numeric_medians_[col])
            if col in self.cap_bounds_:
                lower, upper = self.cap_bounds_[col]
                numeric = numeric.clip(lower=lower, upper=upper)
            if col in self.log_transform_columns_:
                numeric = np.log1p(numeric.clip(lower=0))
            transformed[col] = numeric.astype(float)
        for col in self.binary_categorical_columns:
            transformed[col] = X_norm[col].map(map_binary_clinical_value).astype(str)
        for col in self.demographic_categorical_columns:
            transformed[col] = X_norm[col].map(clean_category_value).fillna(self.categorical_modes_[col]).astype(str)
        for col in self.nonbinary_categorical_columns:
            transformed[col] = X_norm[col].map(clean_category_value).fillna("2").astype(str)
        return transformed[self.feature_order_].copy()

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)


print("Cleaning helpers prepared.")

Cleaning helpers prepared.


## 2.4 Stratified Train-Test Split

The split is performed before fitting imputation, capping, transformations, encoders, scalers, feature selectors, tuning, or threshold selection.

In [9]:
X = model_data[feature_columns].copy()
y = model_data[TARGET_BINARY].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame(
    [
        {"split": "train", "n": len(y_train), "abnormal_n": int(y_train.sum()), "abnormal_pct": float(y_train.mean() * 100)},
        {"split": "held_out_test", "n": len(y_test), "abnormal_n": int(y_test.sum()), "abnormal_pct": float(y_test.mean() * 100)},
    ]
)
print("Stratified split complete. Held-out test data is reserved for final evaluation only.")
display(split_summary)

Stratified split complete. Held-out test data is reserved for final evaluation only.


,split,n,abnormal_n,abnormal_pct
0,train,230,169,73.478261
1,held_out_test,58,42,72.413793


## 2.5 Preprocessing Pipeline Execution

A shared clinical cleaner is fit on the training set, then model-specific matrices are created for linear, tree, and CatBoost models.

In [10]:
def make_ohe_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


class ModelMatrixBuilder:
    def __init__(self, numeric_columns, categorical_columns):
        self.numeric_columns = list(numeric_columns)
        self.categorical_columns = list(categorical_columns)
        self.scaler_ = StandardScaler()
        self.ohe_ = make_ohe_encoder()

    def fit(self, X_clean):
        self.scaler_.fit(X_clean[self.numeric_columns])
        self.ohe_.fit(X_clean[self.categorical_columns])
        self.encoded_categorical_columns_ = list(self.ohe_.get_feature_names_out(self.categorical_columns))
        self.linear_feature_names_ = self.numeric_columns + self.encoded_categorical_columns_
        self.tree_feature_names_ = self.numeric_columns + self.encoded_categorical_columns_
        return self

    def transform_linear(self, X_clean):
        numeric_scaled = pd.DataFrame(
            self.scaler_.transform(X_clean[self.numeric_columns]),
            columns=self.numeric_columns,
            index=X_clean.index,
        )
        categorical_encoded = pd.DataFrame(
            self.ohe_.transform(X_clean[self.categorical_columns]),
            columns=self.encoded_categorical_columns_,
            index=X_clean.index,
        )
        return pd.concat([numeric_scaled, categorical_encoded], axis=1)[self.linear_feature_names_]

    def transform_tree(self, X_clean):
        numeric_unscaled = X_clean[self.numeric_columns].astype(float).copy()
        categorical_encoded = pd.DataFrame(
            self.ohe_.transform(X_clean[self.categorical_columns]),
            columns=self.encoded_categorical_columns_,
            index=X_clean.index,
        )
        return pd.concat([numeric_unscaled, categorical_encoded], axis=1)[self.tree_feature_names_]


def prepare_catboost_frame(X_clean, categorical_columns):
    X_cb = X_clean.copy()
    for col in categorical_columns:
        X_cb[col] = X_cb[col].astype(str)
    return X_cb


print("Model matrix builder prepared.")

Model matrix builder prepared.


In [11]:
clinical_preprocessor = ClinicalPreprocessor(
    numeric_columns=numeric_columns,
    binary_categorical_columns=binary_clinical_categorical_columns,
    demographic_categorical_columns=demographic_categorical_columns,
    nonbinary_categorical_columns=nonbinary_clinical_categorical_columns,
    missing_flag_features=LIPID_MISSING_FLAG_FEATURES,
    capping_candidates=CAPPING_CANDIDATES,
    log_transform_candidates=LOG_TRANSFORM_CANDIDATES,
)

X_train_clean = clinical_preprocessor.fit_transform(X_train_raw, y_train)
X_test_clean = clinical_preprocessor.transform(X_test_raw)

numeric_model_columns = clinical_preprocessor.numeric_model_columns_
categorical_model_columns = clinical_preprocessor.categorical_model_columns_
catboost_categorical_features = list(categorical_model_columns)

matrix_builder = ModelMatrixBuilder(numeric_model_columns, categorical_model_columns).fit(X_train_clean)
X_train_linear = matrix_builder.transform_linear(X_train_clean)
X_test_linear = matrix_builder.transform_linear(X_test_clean)
X_train_tree = matrix_builder.transform_tree(X_train_clean)
X_test_tree = matrix_builder.transform_tree(X_test_clean)
X_train_catboost = prepare_catboost_frame(X_train_clean, catboost_categorical_features)
X_test_catboost = prepare_catboost_frame(X_test_clean, catboost_categorical_features)

preprocessing_summary = pd.DataFrame(
    [
        {"artifact": "clean_train_shape", "value": X_train_clean.shape},
        {"artifact": "linear_train_shape", "value": X_train_linear.shape},
        {"artifact": "tree_train_shape", "value": X_train_tree.shape},
        {"artifact": "catboost_train_shape", "value": X_train_catboost.shape},
        {"artifact": "numeric_model_columns", "value": len(numeric_model_columns)},
        {"artifact": "categorical_model_columns", "value": len(categorical_model_columns)},
        {"artifact": "log_transformed_columns", "value": clinical_preprocessor.log_transform_columns_},
        {"artifact": "train_derived_cap_bounds", "value": clinical_preprocessor.cap_bounds_},
    ]
)
display(preprocessing_summary)

,artifact,value
0,clean_train_shape,"(230, 43)"
1,linear_train_shape,"(230, 96)"
2,tree_train_shape,"(230, 96)"
3,catboost_train_shape,"(230, 43)"
4,numeric_model_columns,20
5,categorical_model_columns,23
6,log_transformed_columns,"[WHITE CELL COUNT, BLOOD UREA, CREATININE, ALA..."
7,train_derived_cap_bounds,"{'PLATELET COUNT': (27.460000000000004, 584.76..."


## 2.6 Feature Selection

Feature selection is performed before benchmarking and uses training data only. Feature exclusion above is not counted as feature selection.

CatBoost uses BorutaSHAP for all-relevant ML risk-factor discovery. Tentative BorutaSHAP features are reported separately and are not included in the main CatBoost model unless explicitly promoted by a predefined rule.

In [12]:
FEATURE_SELECTION_METHODS = {
    "logistic_regression": "L1/Lasso LogisticRegressionCV on scaled one-hot training matrix",
    "linear_svm": "RFECV with LinearSVC on scaled one-hot training matrix",
    "random_forest": "Boruta with RandomForestClassifier on unscaled one-hot training matrix",
    "catboost": "BorutaSHAP all-relevant feature selection with CatBoostClassifier on train-only CatBoost matrix",
}

selected_features = {}
selected_feature_counts = {}
feature_selection_details = []
selected_catboost_categorical_features = []
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

display(pd.DataFrame([{"model": key, "method": value} for key, value in FEATURE_SELECTION_METHODS.items()]))

,model,method
0,logistic_regression,L1/Lasso LogisticRegressionCV on scaled one-ho...
1,linear_svm,RFECV with LinearSVC on scaled one-hot trainin...
2,random_forest,Boruta with RandomForestClassifier on unscaled...
3,catboost,BorutaSHAP all-relevant feature selection with...


In [13]:
l1_selector = LogisticRegressionCV(
    penalty="l1",
    solver="liblinear",
    Cs=np.logspace(-3, 1, 10),
    cv=cv,
    scoring="f1",
    class_weight="balanced",
    max_iter=5000,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
l1_selector.fit(X_train_linear, y_train)
lr_selected = list(X_train_linear.columns[np.abs(l1_selector.coef_).ravel() > 1e-8])
if not lr_selected:
    raise RuntimeError("L1/Lasso Logistic Regression selected zero features; stopping rather than benchmarking full features.")
selected_features["logistic_regression"] = lr_selected
feature_selection_details.append(
    {
        "model": "Logistic Regression",
        "method": FEATURE_SELECTION_METHODS["logistic_regression"],
        "selected_count": len(lr_selected),
        "details": f"Best C={float(l1_selector.C_[0]):.5g}",
    }
)

linear_svm_rfecv = RFECV(
    estimator=LinearSVC(penalty="l2", class_weight="balanced", max_iter=30000, dual="auto", random_state=RANDOM_STATE),
    step=0.1,
    min_features_to_select=min(5, X_train_linear.shape[1]),
    cv=cv,
    scoring="f1",
    n_jobs=1,
)
linear_svm_rfecv.fit(X_train_linear, y_train)
svm_selected = list(X_train_linear.columns[linear_svm_rfecv.support_])
if not svm_selected:
    raise RuntimeError("RFECV Linear SVM selected zero features; stopping rather than benchmarking full features.")
selected_features["linear_svm"] = svm_selected
feature_selection_details.append(
    {
        "model": "Linear SVM",
        "method": FEATURE_SELECTION_METHODS["linear_svm"],
        "selected_count": len(svm_selected),
        "details": f"RFECV selected {linear_svm_rfecv.n_features_} features",
    }
)

rf_for_boruta = RandomForestClassifier(n_estimators=1000, max_depth=5, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1)
boruta_selector = BorutaPy(
    estimator=rf_for_boruta,
    n_estimators="auto",
    perc=100,
    alpha=0.05,
    two_step=True,
    max_iter=100,
    random_state=RANDOM_STATE,
    verbose=0,
)
boruta_selector.fit(X_train_tree.values, y_train.values)
rf_selected_confirmed = list(X_train_tree.columns[boruta_selector.support_])
rf_selected_tentative = list(X_train_tree.columns[boruta_selector.support_weak_])
if not rf_selected_confirmed:
    raise RuntimeError(
        "Boruta selected zero confirmed Random Forest features; stopping rather than benchmarking Random Forest with full features. "
        f"Tentative features: {rf_selected_tentative}"
    )
selected_features["random_forest"] = rf_selected_confirmed
feature_selection_details.append(
    {
        "model": "Random Forest",
        "method": FEATURE_SELECTION_METHODS["random_forest"],
        "selected_count": len(rf_selected_confirmed),
        "details": f"Tentative features: {len(rf_selected_tentative)}",
    }
)

borutashap_catboost_model = CatBoostClassifier(
    iterations=BORUTASHAP_CATBOOST_ITERATIONS,
    depth=4,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="F1",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)

X_train_borutashap = X_train_catboost.reset_index(drop=True).copy()
for col in catboost_categorical_features:
    if col in X_train_borutashap.columns:
        X_train_borutashap[col] = X_train_borutashap[col].astype("category")
y_train_borutashap = y_train.reset_index(drop=True).astype(int)

borutashap_selector = BorutaShap(
    model=borutashap_catboost_model,
    importance_measure="shap",
    classification=True,
    percentile=BORUTASHAP_PERCENTILE,
    pvalue=BORUTASHAP_PVALUE,
)
borutashap_selector.fit(
    X=X_train_borutashap,
    y=y_train_borutashap,
    n_trials=BORUTASHAP_N_TRIALS,
    random_state=RANDOM_STATE,
    sample=False,
    train_or_test="train",
    normalize=True,
    verbose=False,
)

borutashap_accepted_features = [col for col in X_train_catboost.columns if col in set(borutashap_selector.accepted)]
borutashap_tentative_features = [col for col in X_train_catboost.columns if col in set(borutashap_selector.tentative)]
borutashap_rejected_features = [col for col in X_train_catboost.columns if col in set(borutashap_selector.rejected)]
catboost_selected_features = borutashap_accepted_features.copy()
if not catboost_selected_features:
    raise RuntimeError("BorutaSHAP selected zero accepted CatBoost features; tentative features are reported but not auto-included.")

selected_features["catboost"] = catboost_selected_features
selected_catboost_categorical_features = [col for col in catboost_categorical_features if col in catboost_selected_features]

borutashap_importance = borutashap_selector.history_x.drop(columns=["Max_Shadow", "Min_Shadow", "Mean_Shadow", "Median_Shadow"], errors="ignore")
borutashap_importance_summary = pd.DataFrame(
    {
        "feature": list(borutashap_importance.columns),
        "mean_importance": borutashap_importance.mean(axis=0).values,
        "median_importance": borutashap_importance.median(axis=0).values,
        "max_importance": borutashap_importance.max(axis=0).values,
    }
)
borutashap_status_map = {
    **{feature: "accepted" for feature in borutashap_accepted_features},
    **{feature: "tentative" for feature in borutashap_tentative_features},
    **{feature: "rejected" for feature in borutashap_rejected_features},
}
borutashap_feature_status = pd.DataFrame(
    [{"feature": feature, "borutashap_status": borutashap_status_map.get(feature, "unassigned")} for feature in X_train_catboost.columns]
).merge(borutashap_importance_summary, on="feature", how="left")
borutashap_feature_status["used_in_main_catboost"] = borutashap_feature_status["feature"].isin(catboost_selected_features)
borutashap_feature_status = borutashap_feature_status.sort_values(
    ["used_in_main_catboost", "median_importance"], ascending=[False, False]
).reset_index(drop=True)

catboost_final_selected_features_table = pd.DataFrame({"selected_feature": catboost_selected_features})
borutashap_tentative_features_table = pd.DataFrame({"tentative_feature": borutashap_tentative_features})
borutashap_rejected_features_table = pd.DataFrame({"rejected_feature": borutashap_rejected_features})

feature_selection_details.append(
    {
        "model": "CatBoost",
        "method": FEATURE_SELECTION_METHODS["catboost"],
        "selected_count": len(catboost_selected_features),
        "details": f"Accepted={len(borutashap_accepted_features)}; tentative={len(borutashap_tentative_features)}; rejected={len(borutashap_rejected_features)}; selected categorical={len(selected_catboost_categorical_features)}",
    }
)

print(f"BorutaSHAP accepted CatBoost features: {len(borutashap_accepted_features)}")
print(f"BorutaSHAP tentative CatBoost features: {len(borutashap_tentative_features)}")
print(f"BorutaSHAP rejected CatBoost features: {len(borutashap_rejected_features)}")
display(borutashap_feature_status)
display(catboost_final_selected_features_table)
display(borutashap_tentative_features_table)
display(borutashap_rejected_features_table)

selected_feature_counts = {key: len(value) for key, value in selected_features.items()}
feature_selection_summary = pd.DataFrame(feature_selection_details)
selected_feature_table = pd.DataFrame(
    [{"model": model, "selected_feature": feature} for model, features in selected_features.items() for feature in features]
)

display(feature_selection_summary)
display(selected_feature_table)
display(borutashap_feature_status.head(15))


  0%|          | 0/100 [00:00<?, ?it/s]


  1%|          | 1/100 [00:01<02:27,  1.49s/it]


  2%|▏         | 2/100 [00:02<02:23,  1.47s/it]


  3%|▎         | 3/100 [00:04<02:16,  1.41s/it]


  4%|▍         | 4/100 [00:05<02:10,  1.36s/it]


  5%|▌         | 5/100 [00:06<02:07,  1.35s/it]

... (56 outputs left out)

,rejected_feature
0,AGE
1,Systolic
2,Diastolic
3,HAEMOGLOBIN
4,PLATELET COUNT
5,WHITE CELL COUNT
6,POTASSIUM
7,CREATININE
8,TOTAL PROTEIN
9,ALANINE TRANSAMINASE


,model,method,selected_count,details
0,Logistic Regression,L1/Lasso LogisticRegressionCV on scaled one-ho...,51,Best C=1.2915
1,Linear SVM,RFECV with LinearSVC on scaled one-hot trainin...,69,RFECV selected 69 features
2,Random Forest,Boruta with RandomForestClassifier on unscaled...,5,Tentative features: 2
3,CatBoost,BorutaSHAP all-relevant feature selection with...,4,Accepted=4; tentative=1; rejected=38; selected...


,model,selected_feature
0,logistic_regression,AGE
1,logistic_regression,Systolic
2,logistic_regression,Diastolic
3,logistic_regression,HAEMOGLOBIN
4,logistic_regression,PLATELET COUNT
...,...,...
124,random_forest,ALBUMIN
125,catboost,BLOOD UREA
126,catboost,ALBUMIN
127,catboost,SEIZURES


,feature,borutashap_status,mean_importance,median_importance,max_importance,used_in_main_catboost
0,ALBUMIN,accepted,3.131398,2.748697,7.202625,True
1,SEIZURES,accepted,1.303350,1.091818,4.227163,True
2,GENERALISED SEIZURE,accepted,0.597862,0.397845,3.495667,True
3,BLOOD UREA,accepted,0.138613,0.086976,1.454860,True
4,HYPERTENSION,rejected,0.455318,0.449510,0.971001,False
5,WHITE CELL COUNT,rejected,0.248678,0.376391,0.891832,False
6,HAEMOGLOBIN,rejected,0.228111,0.251891,0.566221,False
7,TOTAL PROTEIN,rejected,0.393346,0.238887,1.044722,False
8,POTASSIUM,rejected,0.118291,0.187021,0.358608,False
9,CALCIUM,rejected,0.240270,0.179703,0.798112,False


# 3. Model Benchmarking

This section benchmarks selected-feature models using training-only cross-validation. The held-out test set remains reserved for the final frozen model evaluation.

## 3.1 Evaluation Metrics Definition

F1-score and abnormal EEG recall/sensitivity are primary metrics. Accuracy is reported only as a secondary metric.

In [14]:
def binary_metrics(y_true, y_score, threshold=0.5):
    y_true = pd.Series(y_true).astype(int).to_numpy()
    y_score = np.asarray(y_score, dtype=float)
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    try:
        roc_auc = roc_auc_score(y_true, y_score)
    except ValueError:
        roc_auc = np.nan
    return {
        "threshold": float(threshold),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def selected_matrix(model_key, split):
    if model_key in ["logistic_regression", "linear_svm"]:
        base = X_train_linear if split == "train" else X_test_linear
    elif model_key == "random_forest":
        base = X_train_tree if split == "train" else X_test_tree
    elif model_key == "catboost":
        base = X_train_catboost if split == "train" else X_test_catboost
    else:
        raise KeyError(model_key)
    return base[selected_features[model_key]].copy()


def make_model(model_key, params=None):
    params = params or {}
    if model_key == "logistic_regression":
        return LogisticRegression(C=params.get("C", 1.0), solver="liblinear", class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE)
    if model_key == "linear_svm":
        base_svm = LinearSVC(C=params.get("C", 1.0), penalty="l2", class_weight="balanced", max_iter=30000, dual="auto", random_state=RANDOM_STATE)
        return CalibratedClassifierCV(estimator=base_svm, cv=3)
    if model_key == "random_forest":
        return RandomForestClassifier(
            n_estimators=params.get("n_estimators", 500),
            max_depth=params.get("max_depth", None),
            min_samples_leaf=params.get("min_samples_leaf", 1),
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
    if model_key == "catboost":
        return CatBoostClassifier(
            iterations=params.get("iterations", 200),
            depth=params.get("depth", 4),
            learning_rate=params.get("learning_rate", 0.05),
            l2_leaf_reg=params.get("l2_leaf_reg", 3),
            loss_function="Logloss",
            eval_metric="F1",
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )
    raise KeyError(model_key)


def cat_features_for_matrix(model_key, X_matrix):
    if model_key != "catboost":
        return []
    return [col for col in selected_catboost_categorical_features if col in X_matrix.columns]


def fit_model(model, model_key, X_fit, y_fit):
    if model_key == "catboost":
        model.fit(X_fit, y_fit, cat_features=cat_features_for_matrix(model_key, X_fit), verbose=False)
    else:
        model.fit(X_fit, y_fit)
    return model


def predict_scores(model, X_score):
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X_score))[:, 1]
    decision = np.asarray(model.decision_function(X_score), dtype=float)
    return 1.0 / (1.0 + np.exp(-decision))


def oof_scores(model_key, X_matrix, y_values, params=None):
    y_values = pd.Series(y_values).astype(int)
    scores = pd.Series(index=y_values.index, dtype=float)
    for train_idx, valid_idx in cv.split(X_matrix, y_values):
        model = make_model(model_key, params=params)
        fit_model(model, model_key, X_matrix.iloc[train_idx].copy(), y_values.iloc[train_idx].copy())
        scores.iloc[valid_idx] = predict_scores(model, X_matrix.iloc[valid_idx].copy())
    return scores.astype(float)


metric_priority = ["f1", "recall_sensitivity", "balanced_accuracy"]
print("Metric helpers and model factories are ready.")

Metric helpers and model factories are ready.


## 3.2 Modelling

The benchmark compares only selected-feature models using out-of-fold training predictions.

In [15]:
MODEL_DISPLAY_NAMES = {
    "logistic_regression": "Logistic Regression",
    "linear_svm": "Linear SVM",
    "random_forest": "Random Forest",
    "catboost": "CatBoost",
}

benchmark_rows = []
benchmark_oof_scores = {}
for model_key in ["logistic_regression", "linear_svm", "random_forest", "catboost"]:
    X_model_train = selected_matrix(model_key, "train")
    scores = oof_scores(model_key, X_model_train, y_train)
    benchmark_oof_scores[model_key] = scores
    metrics = binary_metrics(y_train, scores, threshold=0.5)
    metrics.update(
        {
            "model_key": model_key,
            "model": MODEL_DISPLAY_NAMES[model_key],
            "feature_selection_method": FEATURE_SELECTION_METHODS[model_key],
            "selected_feature_count": len(selected_features[model_key]),
        }
    )
    benchmark_rows.append(metrics)

benchmark_results = pd.DataFrame(benchmark_rows)[
    [
        "model_key",
        "model",
        "selected_feature_count",
        "f1",
        "recall_sensitivity",
        "precision",
        "specificity",
        "balanced_accuracy",
        "accuracy",
        "roc_auc",
        "tn",
        "fp",
        "fn",
        "tp",
        "feature_selection_method",
    ]
].sort_values(metric_priority, ascending=False)

FINAL_MODEL_KEY = "catboost"
FINAL_MODEL_NAME = MODEL_DISPLAY_NAMES[FINAL_MODEL_KEY]
FINAL_MODEL_SELECTION_REASON = "CatBoost retained as final model because BorutaSHAP all-relevant ML risk-factor discovery is the project focus."

display(benchmark_results)
print(f"Selected final model for train-only tuning: {FINAL_MODEL_NAME}")
print(FINAL_MODEL_SELECTION_REASON)

,model_key,model,selected_feature_count,f1,recall_sensitivity,precision,specificity,balanced_accuracy,accuracy,roc_auc,tn,fp,fn,tp,feature_selection_method
1,linear_svm,Linear SVM,69,0.861619,0.976331,0.771028,0.196721,0.586526,0.769565,0.757396,12,49,4,165,RFECV with LinearSVC on scaled one-hot trainin...
2,random_forest,Random Forest,5,0.848485,0.911243,0.793814,0.344262,0.627752,0.760870,0.701232,21,40,15,154,Boruta with RandomForestClassifier on unscaled...
3,catboost,CatBoost,4,0.825688,0.798817,0.854430,0.622951,0.710884,0.752174,0.784363,38,23,34,135,BorutaSHAP all-relevant feature selection with...
0,logistic_regression,Logistic Regression,51,0.822430,0.781065,0.868421,0.672131,0.726598,0.752174,0.775051,41,20,37,132,L1/Lasso LogisticRegressionCV on scaled one-ho...


Selected final model for train-only tuning: CatBoost
CatBoost retained as final model because BorutaSHAP all-relevant ML risk-factor discovery is the project focus.


# 4. Final Model Tuning

This section tunes the selected final model on training data only, tunes the threshold with training out-of-fold predictions only, and then evaluates the frozen model once on the held-out test set.

## 4.1 Final Model Tuning

The final model is tuned after feature selection using stratified cross-validation on the training set.

In [16]:
TUNING_GRIDS = {
    "logistic_regression": {"C": [0.1, 1.0, 10.0]},
    "linear_svm": {"C": [0.1, 1.0, 10.0]},
    "random_forest": {"n_estimators": [500], "max_depth": [None, 5], "min_samples_leaf": [1, 3]},
    "catboost": {"iterations": [150, 250], "depth": [3, 4], "learning_rate": [0.03], "l2_leaf_reg": [3]},
}

X_train_final = selected_matrix(FINAL_MODEL_KEY, "train")
X_test_final = selected_matrix(FINAL_MODEL_KEY, "test")

tuning_rows = []
tuning_oof_by_params = []
for params in ParameterGrid(TUNING_GRIDS[FINAL_MODEL_KEY]):
    scores = oof_scores(FINAL_MODEL_KEY, X_train_final, y_train, params=params)
    metrics = binary_metrics(y_train, scores, threshold=0.5)
    row = {"model_key": FINAL_MODEL_KEY, "model": FINAL_MODEL_NAME, "params": params}
    row.update(metrics)
    tuning_rows.append(row)
    tuning_oof_by_params.append((params, scores, metrics))

tuning_results = pd.DataFrame(tuning_rows).sort_values(metric_priority, ascending=False).reset_index(drop=True)
best_tuning_params = tuning_results.iloc[0]["params"]
best_tuning_scores = next(scores for params, scores, _ in tuning_oof_by_params if params == best_tuning_params)

best_final_model = make_model(FINAL_MODEL_KEY, params=best_tuning_params)
fit_model(best_final_model, FINAL_MODEL_KEY, X_train_final, y_train)

print(f"Final model selected for tuning: {FINAL_MODEL_NAME}")
print(f"Best train-CV tuning parameters: {best_tuning_params}")
display(tuning_results)

Final model selected for tuning: CatBoost
Best train-CV tuning parameters: {'depth': 3, 'iterations': 250, 'l2_leaf_reg': 3, 'learning_rate': 0.03}


,model_key,model,params,threshold,f1,recall_sensitivity,precision,specificity,balanced_accuracy,accuracy,roc_auc,tn,fp,fn,tp
0,catboost,CatBoost,"{'depth': 3, 'iterations': 250, 'l2_leaf_reg':...",0.5,0.835913,0.798817,0.876623,0.688525,0.743671,0.769565,0.787370,42,19,34,135
1,catboost,CatBoost,"{'depth': 4, 'iterations': 250, 'l2_leaf_reg':...",0.5,0.835913,0.798817,0.876623,0.688525,0.743671,0.769565,0.787952,42,19,34,135
2,catboost,CatBoost,"{'depth': 4, 'iterations': 150, 'l2_leaf_reg':...",0.5,0.811502,0.751479,0.881944,0.721311,0.736395,0.743478,0.794160,44,17,42,127
3,catboost,CatBoost,"{'depth': 3, 'iterations': 150, 'l2_leaf_reg':...",0.5,0.796117,0.727811,0.878571,0.721311,0.724561,0.726087,0.793772,44,17,46,123


## 4.2 Threshold Tuning

The classification threshold is selected with out-of-fold training predictions only. The held-out test labels are not used for threshold selection.

In [17]:
def threshold_table(y_true, y_score, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.10, 0.91, 0.01), 2)
    return pd.DataFrame([binary_metrics(y_true, y_score, threshold=threshold) for threshold in thresholds])


threshold_results = threshold_table(y_train, best_tuning_scores)
threshold_results = threshold_results.sort_values(metric_priority, ascending=False).reset_index(drop=True)
selected_threshold = float(threshold_results.iloc[0]["threshold"])
threshold_selection_metrics = threshold_results.iloc[0].to_dict()

print(f"Selected threshold from training OOF predictions only: {selected_threshold:.2f}")
display(threshold_results.head(15))

Selected threshold from training OOF predictions only: 0.29


,threshold,f1,recall_sensitivity,precision,specificity,balanced_accuracy,accuracy,roc_auc,tn,fp,fn,tp
0,0.29,0.878187,0.917160,0.842391,0.524590,0.720875,0.813043,0.78737,32,29,14,155
1,0.30,0.877493,0.911243,0.846154,0.540984,0.726113,0.813043,0.78737,33,28,15,154
2,0.26,0.877095,0.928994,0.830688,0.475410,0.702202,0.808696,0.78737,29,32,12,157
3,0.27,0.877095,0.928994,0.830688,0.475410,0.702202,0.808696,0.78737,29,32,12,157
4,0.33,0.876081,0.899408,0.853933,0.573770,0.736589,0.813043,0.78737,35,26,17,152
5,0.28,0.875706,0.917160,0.837838,0.508197,0.712678,0.808696,0.78737,31,30,14,155
6,0.31,0.874286,0.905325,0.845304,0.540984,0.723155,0.808696,0.78737,33,28,16,153
7,0.32,0.873563,0.899408,0.849162,0.557377,0.728393,0.808696,0.78737,34,27,17,152
8,0.25,0.872222,0.928994,0.821990,0.442623,0.685809,0.800000,0.78737,27,34,12,157
9,0.24,0.870523,0.934911,0.814433,0.409836,0.672374,0.795652,0.78737,25,36,11,158


## 4.3 Final Held-Out Test Evaluation

The tuned model and selected threshold are frozen here, then evaluated once on the held-out test set.

In [18]:
test_scores = predict_scores(best_final_model, X_test_final)
final_test_metrics = binary_metrics(y_test, test_scores, threshold=selected_threshold)
final_test_results = pd.DataFrame([{**{"model": FINAL_MODEL_NAME, "params": best_tuning_params}, **final_test_metrics}])

previous_catboost_reference = {
    "selected_feature_count": 10,
    "threshold": 0.30,
    "f1": 0.835,
    "recall_sensitivity": 0.905,
    "precision": 0.776,
    "specificity": 0.312,
    "balanced_accuracy": 0.609,
    "accuracy": 0.741,
    "roc_auc": 0.766,
    "tn": 5,
    "fp": 11,
    "fn": 4,
    "tp": 38,
}
current_catboost_summary = {
    "selected_feature_count": len(catboost_selected_features),
    **final_test_metrics,
}
catboost_metric_changes_vs_previous = pd.DataFrame(
    [
        {
            "metric": metric,
            "previous_catboost": previous_value,
            "current_catboost": current_catboost_summary.get(metric, selected_threshold if metric == "threshold" else np.nan),
            "change": current_catboost_summary.get(metric, selected_threshold if metric == "threshold" else np.nan) - previous_value,
        }
        for metric, previous_value in previous_catboost_reference.items()
    ]
)
final_confusion_matrix = pd.DataFrame(
    [[final_test_metrics["tn"], final_test_metrics["fp"]], [final_test_metrics["fn"], final_test_metrics["tp"]]],
    index=["Actual normal", "Actual abnormal"],
    columns=["Predicted normal", "Predicted abnormal"],
)

print("Final held-out test evaluation complete. This is the first metric evaluation using held-out test labels.")
display(final_test_results)
display(final_confusion_matrix)
print("Metric changes versus previous CatBoost reference result:")
display(catboost_metric_changes_vs_previous)

final_test_results.to_csv(OUTPUT_DIR / "final_held_out_test_metrics.csv", index=False)
benchmark_results.to_csv(OUTPUT_DIR / "benchmark_results_train_cv.csv", index=False)
feature_selection_summary.to_csv(OUTPUT_DIR / "feature_selection_summary.csv", index=False)
selected_feature_table.to_csv(OUTPUT_DIR / "selected_features.csv", index=False)
borutashap_feature_status.to_csv(OUTPUT_DIR / "catboost_borutashap_feature_status.csv", index=False)
catboost_final_selected_features_table.to_csv(OUTPUT_DIR / "catboost_borutashap_accepted_features.csv", index=False)
borutashap_tentative_features_table.to_csv(OUTPUT_DIR / "catboost_borutashap_tentative_features.csv", index=False)
borutashap_rejected_features_table.to_csv(OUTPUT_DIR / "catboost_borutashap_rejected_features.csv", index=False)
catboost_metric_changes_vs_previous.to_csv(OUTPUT_DIR / "catboost_metric_changes_vs_previous.csv", index=False)

Final held-out test evaluation complete. This is the first metric evaluation using held-out test labels.


,model,params,threshold,f1,recall_sensitivity,precision,specificity,balanced_accuracy,accuracy,roc_auc,tn,fp,fn,tp
0,CatBoost,"{'depth': 3, 'iterations': 250, 'l2_leaf_reg':...",0.29,0.853933,0.904762,0.808511,0.4375,0.671131,0.775862,0.809524,7,9,4,38


,Predicted normal,Predicted abnormal
Actual normal,7,9
Actual abnormal,4,38


Metric changes versus previous CatBoost reference result:


,metric,previous_catboost,current_catboost,change
0,selected_feature_count,10.000,4.000000,-6.000000
1,threshold,0.300,0.290000,-0.010000
2,f1,0.835,0.853933,0.018933
3,recall_sensitivity,0.905,0.904762,-0.000238
4,precision,0.776,0.808511,0.032511
5,specificity,0.312,0.437500,0.125500
6,balanced_accuracy,0.609,0.671131,0.062131
7,accuracy,0.741,0.775862,0.034862
8,roc_auc,0.766,0.809524,0.043524
9,tn,5.000,7.000000,2.000000


# 5. SHAP Analysis

This section explains the frozen final model. SHAP explains model behavior, not causality.

## 5.1 Global SHAP Explanation

Global SHAP bar and beeswarm plots are generated for the frozen CatBoost model using the BorutaSHAP accepted feature set. BorutaSHAP was used for all-relevant ML risk-factor discovery; SHAP explains how the final CatBoost model used those selected features.

In [19]:
def numeric_display_frame(X_frame):
    display_frame = X_frame.copy()
    category_maps = {}
    for col in display_frame.columns:
        if not pd.api.types.is_numeric_dtype(display_frame[col]):
            cat = pd.Categorical(display_frame[col].astype(str))
            category_maps[col] = dict(enumerate(cat.categories))
            display_frame[col] = cat.codes.astype(float)
    return display_frame.astype(float), category_maps


def build_shap_explanation(model_key, model, X_background, X_explain):
    feature_names = list(X_explain.columns)
    X_display, category_maps = numeric_display_frame(X_explain)
    if model_key == "catboost":
        pool = Pool(X_explain, cat_features=cat_features_for_matrix(model_key, X_explain))
        raw_shap = model.get_feature_importance(pool, type="ShapValues")
        explanation = shap.Explanation(
            values=raw_shap[:, :-1],
            base_values=raw_shap[:, -1],
            data=X_display.to_numpy(),
            feature_names=feature_names,
        )
        return explanation, category_maps

    if model_key == "random_forest":
        explainer = shap.TreeExplainer(model)
        raw = explainer.shap_values(X_explain)
        if isinstance(raw, list):
            values = raw[1]
            base = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
        elif np.asarray(raw).ndim == 3:
            values = np.asarray(raw)[:, :, 1]
            base = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value
        else:
            values = np.asarray(raw)
            base = explainer.expected_value
        explanation = shap.Explanation(
            values=values,
            base_values=np.repeat(base, len(X_explain)),
            data=X_display.to_numpy(),
            feature_names=feature_names,
        )
        return explanation, category_maps

    background = X_background.sample(min(100, len(X_background)), random_state=RANDOM_STATE)

    def predict_positive(values):
        return predict_scores(model, pd.DataFrame(values, columns=feature_names))

    explainer = shap.Explainer(
        predict_positive,
        shap.maskers.Independent(background),
        algorithm="permutation",
        feature_names=feature_names,
    )
    explanation = explainer(X_explain, max_evals=max(2 * len(feature_names) + 1, 50))
    values = explanation.values
    base_values = explanation.base_values
    if np.asarray(values).ndim == 3:
        values = np.asarray(values)[:, :, 1]
        base_values = np.asarray(base_values)[:, 1]
    return shap.Explanation(values=values, base_values=base_values, data=X_display.to_numpy(), feature_names=feature_names), category_maps


X_shap_background = X_train_final.copy()
X_shap_explain = X_test_final.copy()
shap_explanation, shap_category_maps = build_shap_explanation(FINAL_MODEL_KEY, best_final_model, X_shap_background, X_shap_explain)

shap_importance = pd.DataFrame(
    {"feature": list(X_shap_explain.columns), "mean_abs_shap": np.abs(shap_explanation.values).mean(axis=0)}
).sort_values("mean_abs_shap", ascending=False)
top_shap_features = list(shap_importance.head(5)["feature"])

plt.figure(figsize=(8, 6))
shap.plots.bar(shap_explanation, max_display=min(15, len(X_shap_explain.columns)), show=False)
plt.tight_layout()
shap_bar_path = SHAP_OUTPUT_DIR / "global_shap_bar.png"
plt.savefig(shap_bar_path, dpi=200, bbox_inches="tight")
plt.close()

plt.figure(figsize=(8, 6))
shap.plots.beeswarm(shap_explanation, max_display=min(15, len(X_shap_explain.columns)), show=False)
plt.tight_layout()
shap_beeswarm_path = SHAP_OUTPUT_DIR / "global_shap_beeswarm.png"
plt.savefig(shap_beeswarm_path, dpi=200, bbox_inches="tight")
plt.close()

shap_importance.to_csv(SHAP_OUTPUT_DIR / "shap_feature_importance.csv", index=False)
print("Global SHAP outputs produced:")
print(shap_bar_path)
print(shap_beeswarm_path)
display(shap_importance.head(15))

Global SHAP outputs produced:
Z:\R&D\EEG\outputs\shap\global_shap_bar.png
Z:\R&D\EEG\outputs\shap\global_shap_beeswarm.png


,feature,mean_abs_shap
1,ALBUMIN,0.878920
2,SEIZURES,0.568167
0,BLOOD UREA,0.429487
3,GENERALISED SEIZURE,0.357229


## 5.2 SHAP Dependence Plots

Dependence plots are generated for the top SHAP-ranked BorutaSHAP-selected features where feasible. These are model-associated predictors/risk factors, not causal claims.

In [20]:
dependence_plot_paths = []
shap_display_df = pd.DataFrame(shap_explanation.data, columns=list(X_shap_explain.columns), index=X_shap_explain.index)

for feature in top_shap_features[:5]:
    try:
        plt.figure(figsize=(7, 5))
        shap.dependence_plot(
            feature,
            shap_explanation.values,
            shap_display_df,
            feature_names=list(X_shap_explain.columns),
            show=False,
            interaction_index=None,
        )
        plt.tight_layout()
        safe_feature = "".join(ch if ch.isalnum() else "_" for ch in feature).strip("_")[:80]
        path = SHAP_OUTPUT_DIR / f"dependence_{safe_feature}.png"
        plt.savefig(path, dpi=200, bbox_inches="tight")
        plt.close()
        dependence_plot_paths.append(path)
    except Exception as exc:
        plt.close()
        print(f"Skipped dependence plot for {feature}: {exc}")

print("Dependence plot outputs:")
for path in dependence_plot_paths:
    print(path)

Dependence plot outputs:
Z:\R&D\EEG\outputs\shap\dependence_ALBUMIN.png
Z:\R&D\EEG\outputs\shap\dependence_SEIZURES.png
Z:\R&D\EEG\outputs\shap\dependence_BLOOD_UREA.png
Z:\R&D\EEG\outputs\shap\dependence_GENERALISED_SEIZURE.png


## 5.3 Local SHAP Explanation

A de-identified local waterfall plot is generated for one held-out example where feasible. No direct identifiers are included. The local explanation describes model behavior for the selected features, not clinical causality.

In [21]:
waterfall_path = None
try:
    plt.figure(figsize=(8, 6))
    shap.plots.waterfall(shap_explanation[0], max_display=min(15, len(X_shap_explain.columns)), show=False)
    plt.tight_layout()
    waterfall_path = SHAP_OUTPUT_DIR / "local_waterfall_example_held_out_case.png"
    plt.savefig(waterfall_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Local waterfall output: {waterfall_path}")
except Exception as exc:
    plt.close()
    print(f"Waterfall plot was not feasible: {exc}")

Local waterfall output: Z:\R&D\EEG\outputs\shap\local_waterfall_example_held_out_case.png


## 5.4 Clinical Interpretation, Limitations, and Final Summary

This final summary interprets model-associated predictors cautiously for research use only and compares BorutaSHAP-selected ML factors against manuscript-identified risk factors.

In [22]:
shap_outputs = {
    "global_bar": str(shap_bar_path),
    "global_beeswarm": str(shap_beeswarm_path),
    "dependence_plots": [str(path) for path in dependence_plot_paths],
    "waterfall": str(waterfall_path) if waterfall_path else None,
    "importance_csv": str(SHAP_OUTPUT_DIR / "shap_feature_importance.csv"),
}

final_summary = {
    "project": PROJECT_NAME,
    "final_model": FINAL_MODEL_NAME,
    "selected_threshold": selected_threshold,
    "selected_feature_count": len(selected_features[FINAL_MODEL_KEY]),
    "borutashap_accepted_features": borutashap_accepted_features,
    "borutashap_tentative_features": borutashap_tentative_features,
    "final_test_f1": final_test_metrics["f1"],
    "final_test_recall_sensitivity": final_test_metrics["recall_sensitivity"],
    "final_test_precision": final_test_metrics["precision"],
    "final_test_specificity": final_test_metrics["specificity"],
    "final_test_balanced_accuracy": final_test_metrics["balanced_accuracy"],
    "final_test_roc_auc": final_test_metrics["roc_auc"],
    "top_shap_features": top_shap_features,
    "research_only_note": RESEARCH_ONLY_NOTE,
}

manuscript_risk_factor_mapping = {
    "seizures": ["SEIZURES", "FOCAL SEIZURE", "GENERALISED SEIZURE"],
    "hyperlipidaemia": ["HYPERLIPIDEMIA"],
    "infection": ["WHITE CELL COUNT"],
    "structural causes": ["RADIOLOGICAL IMAGING", "BRAIN TUMOUR", "STROKE"],
    "stroke": ["STROKE"],
    "radiological infarct": ["RADIOLOGICAL IMAGING"],
    "creatinine": ["CREATININE"],
    "albumin": ["ALBUMIN"],
    "length of hospitalisation": ["LENGTH OF HOSPITALISATION"],
}

manuscript_risk_factor_comparison = pd.DataFrame(
    [
        {
            "manuscript_risk_factor": risk_factor,
            "candidate_or_related_features": features,
            "borutashap_accepted_overlap": [feature for feature in features if feature in borutashap_accepted_features],
            "borutashap_tentative_overlap": [feature for feature in features if feature in borutashap_tentative_features],
            "borutashap_rejected_overlap": [feature for feature in features if feature in borutashap_rejected_features],
            "notes": "Excluded post-outcome leakage" if "LENGTH OF HOSPITALISATION" in features else "Compared against retained candidate feature pool",
        }
        for risk_factor, features in manuscript_risk_factor_mapping.items()
    ]
)

clinical_caveats = [
    "BorutaSHAP was used for all-relevant ML risk-factor discovery with CatBoost on training data only.",
    "SHAP explains how the final CatBoost model used the BorutaSHAP-selected features; it does not establish causality.",
    "Selected features are model-associated predictors/risk factors, not causal claims or diagnostic evidence by themselves.",
    "Seizure-related predictors require timing confirmation relative to EEG ordering.",
    "Blood Sugar Control availability and clinical meaning require confirmation.",
    "Missingness flags may reflect testing or documentation patterns rather than biology.",
    "The dataset is small, single-centre, retrospective, and elderly; external validation is required before any clinical use.",
    "No clinical deployment, diagnostic decision support, or treatment recommendation claims are made.",
]

pd.Series(final_summary).to_frame("value").to_csv(OUTPUT_DIR / "final_summary.csv")
manuscript_risk_factor_comparison.to_csv(OUTPUT_DIR / "manuscript_vs_borutashap_risk_factor_comparison.csv", index=False)
pd.Series(shap_outputs).to_frame("path").to_csv(SHAP_OUTPUT_DIR / "shap_outputs.csv")

print("Final model summary")
display(pd.Series(final_summary).to_frame("value"))
print("SHAP outputs")
display(pd.Series(shap_outputs).to_frame("path"))
print("Manuscript risk-factor comparison")
display(manuscript_risk_factor_comparison)
print("Clinical caveats and limitations")
for caveat in clinical_caveats:
    print(f"- {caveat}")

Final model summary


,value
project,EEG Abnormality Prediction
final_model,CatBoost
selected_threshold,0.29
selected_feature_count,4
borutashap_accepted_features,"[BLOOD UREA, ALBUMIN, SEIZURES, GENERALISED SE..."
borutashap_tentative_features,[SODIUM]
final_test_f1,0.853933
final_test_recall_sensitivity,0.904762
final_test_precision,0.808511
final_test_specificity,0.4375


SHAP outputs


,path
global_bar,Z:\R&D\EEG\outputs\shap\global_shap_bar.png
global_beeswarm,Z:\R&D\EEG\outputs\shap\global_shap_beeswarm.png
dependence_plots,[Z:\R&D\EEG\outputs\shap\dependence_ALBUMIN.pn...
waterfall,Z:\R&D\EEG\outputs\shap\local_waterfall_exampl...
importance_csv,Z:\R&D\EEG\outputs\shap\shap_feature_importanc...


Manuscript risk-factor comparison


,manuscript_risk_factor,candidate_or_related_features,borutashap_accepted_overlap,borutashap_tentative_overlap,borutashap_rejected_overlap,notes
0,seizures,"[SEIZURES, FOCAL SEIZURE, GENERALISED SEIZURE]","[SEIZURES, GENERALISED SEIZURE]",[],[FOCAL SEIZURE],Compared against retained candidate feature pool
1,hyperlipidaemia,[HYPERLIPIDEMIA],[],[],[HYPERLIPIDEMIA],Compared against retained candidate feature pool
2,infection,[WHITE CELL COUNT],[],[],[WHITE CELL COUNT],Compared against retained candidate feature pool
3,structural causes,"[RADIOLOGICAL IMAGING, BRAIN TUMOUR, STROKE]",[],[],"[RADIOLOGICAL IMAGING, BRAIN TUMOUR, STROKE]",Compared against retained candidate feature pool
4,stroke,[STROKE],[],[],[STROKE],Compared against retained candidate feature pool
5,radiological infarct,[RADIOLOGICAL IMAGING],[],[],[RADIOLOGICAL IMAGING],Compared against retained candidate feature pool
6,creatinine,[CREATININE],[],[],[CREATININE],Compared against retained candidate feature pool
7,albumin,[ALBUMIN],[ALBUMIN],[],[],Compared against retained candidate feature pool
8,length of hospitalisation,[LENGTH OF HOSPITALISATION],[],[],[],Excluded post-outcome leakage


Clinical caveats and limitations
- BorutaSHAP was used for all-relevant ML risk-factor discovery with CatBoost on training data only.
- SHAP explains how the final CatBoost model used the BorutaSHAP-selected features; it does not establish causality.
- Selected features are model-associated predictors/risk factors, not causal claims or diagnostic evidence by themselves.
- Seizure-related predictors require timing confirmation relative to EEG ordering.
- Blood Sugar Control availability and clinical meaning require confirmation.
- Missingness flags may reflect testing or documentation patterns rather than biology.
- The dataset is small, single-centre, retrospective, and elderly; external validation is required before any clinical use.
- No clinical deployment, diagnostic decision support, or treatment recommendation claims are made.


# 6. Sodium Analysis

This exploratory section analyzes `SODIUM` separately because prior CatBoost feature selection selected it and current BorutaSHAP marked it as tentative. The main final CatBoost model remains unchanged and continues to use BorutaSHAP accepted features only.

### 6.1 Sodium Distribution by EEG Abnormality

This subsection describes the univariate sodium distribution by `EEG_ABNORMAL`, including median, IQR, mean, standard deviation, missing count, and class-stratified visualizations.

In [23]:
SODIUM_OUTPUT_DIR = OUTPUT_DIR / "sodium_analysis"
SODIUM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "SODIUM" not in model_data.columns:
    raise KeyError("SODIUM is not available in the retained modelling dataset.")

sodium_raw = model_data[[TARGET_BINARY, "SODIUM"]].copy()
sodium_raw["SODIUM"] = coerce_numeric(sodium_raw["SODIUM"])
sodium_raw["EEG class"] = np.where(sodium_raw[TARGET_BINARY].astype(int) == 1, "Abnormal EEG", "Normal EEG")

sodium_distribution = (
    sodium_raw.groupby("EEG class", observed=False)["SODIUM"]
    .agg(
        n="size",
        observed="count",
        missing_count=lambda s: int(s.isna().sum()),
        median="median",
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75),
        mean="mean",
        std="std",
    )
    .reset_index()
)
sodium_distribution["IQR"] = sodium_distribution["q3"] - sodium_distribution["q1"]
sodium_distribution = sodium_distribution[
    ["EEG class", "n", "observed", "missing_count", "median", "q1", "q3", "IQR", "mean", "std"]
].round(3)

print("Sodium distribution by EEG_ABNORMAL class")
display(sodium_distribution)
sodium_distribution.to_csv(SODIUM_OUTPUT_DIR / "sodium_distribution_by_eeg_abnormal.csv", index=False)

plot_df = sodium_raw.dropna(subset=["SODIUM"]).copy()

fig, ax = plt.subplots(figsize=(7, 5))
sns.violinplot(data=plot_df, x="EEG class", y="SODIUM", inner="box", cut=0, ax=ax)
ax.axhspan(135, 145, color="tab:green", alpha=0.08, label="Normal sodium range 135-145")
ax.set_title("Sodium distribution by EEG abnormality class")
ax.set_xlabel("")
ax.set_ylabel("Sodium (mmol/L)")
ax.legend(loc="best")
plt.tight_layout()
sodium_violin_path = SODIUM_OUTPUT_DIR / "sodium_violin_by_eeg_abnormal.png"
plt.savefig(sodium_violin_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data=plot_df, x="SODIUM", hue="EEG class", bins=20, kde=True, stat="density", common_norm=False, alpha=0.35, ax=ax)
ax.axvline(135, color="tab:orange", linestyle="--", linewidth=1.5, label="135 mmol/L")
ax.axvline(145, color="tab:red", linestyle="--", linewidth=1.5, label="145 mmol/L")
ax.set_title("Sodium histogram by EEG abnormality class")
ax.set_xlabel("Sodium (mmol/L)")
plt.tight_layout()
sodium_hist_path = SODIUM_OUTPUT_DIR / "sodium_histogram_by_eeg_abnormal.png"
plt.savefig(sodium_hist_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Saved sodium distribution plots to: {SODIUM_OUTPUT_DIR}")

Sodium distribution by EEG_ABNORMAL class


,EEG class,n,observed,missing_count,median,q1,q3,IQR,mean,std
0,Abnormal EEG,211,210,1,139.0,135.0,142.0,7.0,139.343,6.007
1,Normal EEG,77,76,1,139.0,137.0,141.0,4.0,139.329,3.835


Saved sodium distribution plots to: Z:\R&D\EEG\outputs\sodium_analysis


### 6.2 Clinical Sodium Ranges

Sodium is grouped using common clinical cutoffs: hyponatremia `<135`, normal `135-145`, and hypernatremia `>145`. These summaries are descriptive and do not change the final model.

In [24]:
def sodium_range_group(value):
    if pd.isna(value):
        return "Missing"
    if value < 135:
        return "Hyponatremia (<135)"
    if value <= 145:
        return "Normal (135-145)"
    return "Hypernatremia (>145)"

sodium_range_order = ["Hyponatremia (<135)", "Normal (135-145)", "Hypernatremia (>145)", "Missing"]
sodium_ranges = sodium_raw.copy()
sodium_ranges["sodium_group"] = pd.Categorical(
    sodium_ranges["SODIUM"].apply(sodium_range_group),
    categories=sodium_range_order,
    ordered=True,
)

sodium_range_summary = (
    sodium_ranges.groupby("sodium_group", observed=False)
    .agg(
        n=(TARGET_BINARY, "size"),
        abnormal_eeg_count=(TARGET_BINARY, "sum"),
        abnormal_eeg_rate=(TARGET_BINARY, "mean"),
    )
    .reset_index()
)
sodium_range_summary["abnormal_eeg_count"] = sodium_range_summary["abnormal_eeg_count"].astype(int)
sodium_range_summary["abnormal_eeg_rate_pct"] = (sodium_range_summary["abnormal_eeg_rate"] * 100).round(2)

print("Abnormal EEG rate by clinical sodium range")
display(sodium_range_summary[["sodium_group", "n", "abnormal_eeg_count", "abnormal_eeg_rate_pct"]])
sodium_range_summary.to_csv(SODIUM_OUTPUT_DIR / "sodium_clinical_range_abnormal_rate.csv", index=False)


Abnormal EEG rate by clinical sodium range


,sodium_group,n,abnormal_eeg_count,abnormal_eeg_rate_pct
0,Hyponatremia (<135),46,41,89.13
1,Normal (135-145),214,145,67.76
2,Hypernatremia (>145),26,24,92.31
3,Missing,2,1,50.00


### 6.3 Sodium Model Signal

This subsection fits a sensitivity CatBoost model using BorutaSHAP accepted features plus `SODIUM`. It is used only to inspect sodium signal and SHAP dependence; it does not replace or alter the main final CatBoost model.

In [25]:
if "SODIUM" not in X_train_catboost.columns:
    raise KeyError("SODIUM is not present after preprocessing; sensitivity model cannot be fitted.")

sodium_sensitivity_features = list(dict.fromkeys(borutashap_accepted_features + ["SODIUM"]))
main_accepted_features_for_sodium_section = borutashap_accepted_features.copy()

X_train_sodium_sensitivity = X_train_catboost[sodium_sensitivity_features].copy()
X_test_sodium_sensitivity = X_test_catboost[sodium_sensitivity_features].copy()

sodium_sensitivity_model = make_model("catboost", params=best_tuning_params)
fit_model(sodium_sensitivity_model, "catboost", X_train_sodium_sensitivity, y_train)

sodium_shap_explanation, sodium_shap_category_maps = build_shap_explanation(
    "catboost",
    sodium_sensitivity_model,
    X_train_sodium_sensitivity,
    X_test_sodium_sensitivity,
)
sodium_shap_display_df = pd.DataFrame(
    sodium_shap_explanation.data,
    columns=list(X_test_sodium_sensitivity.columns),
    index=X_test_sodium_sensitivity.index,
)

sodium_feature_index = list(X_test_sodium_sensitivity.columns).index("SODIUM")
sodium_shap_values = sodium_shap_explanation.values[:, sodium_feature_index]
sodium_shap_signal_summary = pd.DataFrame(
    [
        {
            "feature": "SODIUM",
            "mean_abs_shap": float(np.abs(sodium_shap_values).mean()),
            "mean_shap": float(np.mean(sodium_shap_values)),
            "min_shap": float(np.min(sodium_shap_values)),
            "max_shap": float(np.max(sodium_shap_values)),
        }
    ]
)

plt.figure(figsize=(7, 5))
shap.dependence_plot(
    "SODIUM",
    sodium_shap_explanation.values,
    sodium_shap_display_df,
    feature_names=list(X_test_sodium_sensitivity.columns),
    show=False,
    interaction_index=None,
)
plt.title("SODIUM SHAP dependence - sensitivity CatBoost model")
plt.tight_layout()
sodium_dependence_path = SODIUM_OUTPUT_DIR / "sodium_shap_dependence_sensitivity_catboost.png"
plt.savefig(sodium_dependence_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close()

sodium_bin_edges = [-np.inf, 130, 135, 140, 145, np.inf]
sodium_bin_labels = ["<130", "130-134", "135-140", "141-145", ">145"]
sodium_binned = sodium_raw.dropna(subset=["SODIUM"]).copy()
sodium_binned["sodium_bin"] = pd.cut(sodium_binned["SODIUM"], bins=sodium_bin_edges, labels=sodium_bin_labels)
sodium_bin_summary = (
    sodium_binned.groupby("sodium_bin", observed=False)
    .agg(n=(TARGET_BINARY, "size"), abnormal_eeg_count=(TARGET_BINARY, "sum"), abnormal_eeg_rate=(TARGET_BINARY, "mean"))
    .reset_index()
)
sodium_bin_summary["abnormal_eeg_count"] = sodium_bin_summary["abnormal_eeg_count"].astype(int)
sodium_bin_summary["abnormal_eeg_rate_pct"] = (sodium_bin_summary["abnormal_eeg_rate"] * 100).round(2)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=sodium_bin_summary, x="sodium_bin", y="abnormal_eeg_rate_pct", color="steelblue", ax=ax)
ax.set_title("Abnormal EEG rate across sodium bins")
ax.set_xlabel("Sodium bin (mmol/L)")
ax.set_ylabel("Abnormal EEG rate (%)")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", fontsize=9)
plt.tight_layout()
sodium_bin_rate_path = SODIUM_OUTPUT_DIR / "sodium_binned_abnormal_eeg_rate.png"
plt.savefig(sodium_bin_rate_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Sodium sensitivity model features:")
print(sodium_sensitivity_features)
print("Sodium SHAP signal summary:")
display(sodium_shap_signal_summary.round(4))
print("Binned abnormal EEG rate across sodium ranges:")
display(sodium_bin_summary)

sodium_shap_signal_summary.to_csv(SODIUM_OUTPUT_DIR / "sodium_shap_signal_summary.csv", index=False)
sodium_bin_summary.to_csv(SODIUM_OUTPUT_DIR / "sodium_binned_abnormal_rate.csv", index=False)
print(f"Saved sodium SHAP dependence plot: {sodium_dependence_path}")
print(f"Saved sodium binned rate plot: {sodium_bin_rate_path}")


Sodium sensitivity model features:
['BLOOD UREA', 'ALBUMIN', 'SEIZURES', 'GENERALISED SEIZURE', 'SODIUM']
Sodium SHAP signal summary:


,feature,mean_abs_shap,mean_shap,min_shap,max_shap
0,SODIUM,0.4834,0.0604,-1.1461,1.7115


Binned abnormal EEG rate across sodium ranges:


,sodium_bin,n,abnormal_eeg_count,abnormal_eeg_rate,abnormal_eeg_rate_pct
0,<130,11,10,0.909091,90.91
1,130-134,54,47,0.870370,87.04
2,135-140,114,70,0.614035,61.40
3,141-145,81,59,0.728395,72.84
4,>145,26,24,0.923077,92.31


Saved sodium SHAP dependence plot: Z:\R&D\EEG\outputs\sodium_analysis\sodium_shap_dependence_sensitivity_catboost.png
Saved sodium binned rate plot: Z:\R&D\EEG\outputs\sodium_analysis\sodium_binned_abnormal_eeg_rate.png


### 6.4 Sensitivity Model Comparison

The main model and sodium sensitivity model are compared using the same train/test split. Thresholds are selected from out-of-fold training predictions only; the held-out test set is used only for final comparison.

In [26]:
def oof_scores_for_catboost_feature_set(features, params=None):
    X_matrix = X_train_catboost[features].copy()
    y_values = pd.Series(y_train).astype(int)
    scores = pd.Series(index=y_values.index, dtype=float)
    candidate_categorical = [col for col in catboost_categorical_features if col in features]
    for train_idx, valid_idx in cv.split(X_matrix, y_values):
        model = make_model("catboost", params=params)
        model.fit(
            X_matrix.iloc[train_idx].copy(),
            y_values.iloc[train_idx].copy(),
            cat_features=[col for col in candidate_categorical if col in X_matrix.columns],
            verbose=False,
        )
        scores.iloc[valid_idx] = predict_scores(model, X_matrix.iloc[valid_idx].copy())
    return scores.astype(float)


def fit_catboost_feature_set(features, params):
    X_matrix = X_train_catboost[features].copy()
    model = make_model("catboost", params=params)
    model.fit(
        X_matrix,
        y_train,
        cat_features=[col for col in catboost_categorical_features if col in features],
        verbose=False,
    )
    return model


def evaluate_catboost_sensitivity_feature_set(model_label, features):
    tuning_rows = []
    tuning_oof_by_params = []
    for params in ParameterGrid(TUNING_GRIDS["catboost"]):
        train_oof_scores = oof_scores_for_catboost_feature_set(features, params=params)
        row = {"params": params}
        row.update(binary_metrics(y_train, train_oof_scores, threshold=0.5))
        tuning_rows.append(row)
        tuning_oof_by_params.append((params, train_oof_scores))

    tuning_results = pd.DataFrame(tuning_rows).sort_values(metric_priority, ascending=False).reset_index(drop=True)
    best_params = tuning_results.iloc[0]["params"]
    best_train_oof_scores = next(scores for params, scores in tuning_oof_by_params if params == best_params)
    train_threshold_table = threshold_table(y_train, best_train_oof_scores).sort_values(metric_priority, ascending=False).reset_index(drop=True)
    tuned_threshold = float(train_threshold_table.iloc[0]["threshold"])

    fitted_model = fit_catboost_feature_set(features, best_params)
    held_out_scores = predict_scores(fitted_model, X_test_catboost[features].copy())
    held_out_metrics = binary_metrics(y_test, held_out_scores, threshold=tuned_threshold)

    row = {
        "model": model_label,
        "feature_count": len(features),
        "includes_sodium": "SODIUM" in features,
        "best_params": best_params,
        "threshold": tuned_threshold,
        **held_out_metrics,
    }
    matrix = pd.DataFrame(
        [[held_out_metrics["tn"], held_out_metrics["fp"]], [held_out_metrics["fn"], held_out_metrics["tp"]]],
        index=["Actual normal", "Actual abnormal"],
        columns=["Predicted normal", "Predicted abnormal"],
    )
    return row, matrix, train_threshold_table, tuning_results, fitted_model, held_out_scores

main_sodium_section_row, main_sodium_section_cm, main_sodium_threshold_table, main_sodium_tuning_results, main_sodium_section_model, main_sodium_scores = evaluate_catboost_sensitivity_feature_set(
    "Main CatBoost: BorutaSHAP accepted features only",
    main_accepted_features_for_sodium_section,
)
sodium_sensitivity_row, sodium_sensitivity_cm, sodium_sensitivity_threshold_table, sodium_sensitivity_tuning_results, sodium_sensitivity_eval_model, sodium_sensitivity_scores = evaluate_catboost_sensitivity_feature_set(
    "Sodium sensitivity CatBoost: accepted features + SODIUM",
    sodium_sensitivity_features,
)

sodium_sensitivity_comparison = pd.DataFrame([main_sodium_section_row, sodium_sensitivity_row])[
    [
        "model",
        "feature_count",
        "includes_sodium",
        "best_params",
        "threshold",
        "f1",
        "recall_sensitivity",
        "precision",
        "specificity",
        "balanced_accuracy",
        "accuracy",
        "roc_auc",
        "tn",
        "fp",
        "fn",
        "tp",
    ]
]

sodium_metric_delta = (
    sodium_sensitivity_comparison.set_index("model")
    .loc["Sodium sensitivity CatBoost: accepted features + SODIUM", ["f1", "recall_sensitivity", "precision", "specificity", "balanced_accuracy", "accuracy", "roc_auc"]]
    - sodium_sensitivity_comparison.set_index("model")
    .loc["Main CatBoost: BorutaSHAP accepted features only", ["f1", "recall_sensitivity", "precision", "specificity", "balanced_accuracy", "accuracy", "roc_auc"]]
).to_frame("sodium_minus_main")

sensitivity_tuning_results = pd.concat(
    [
        main_sodium_tuning_results.assign(model="Main CatBoost: BorutaSHAP accepted features only"),
        sodium_sensitivity_tuning_results.assign(model="Sodium sensitivity CatBoost: accepted features + SODIUM"),
    ],
    ignore_index=True,
)

print("Held-out sensitivity comparison. Hyperparameters and thresholds were selected using training OOF predictions only.")
display(sodium_sensitivity_comparison.round(4))
print("Main model confusion matrix")
display(main_sodium_section_cm)
print("Sodium sensitivity model confusion matrix")
display(sodium_sensitivity_cm)
print("Sensitivity minus main metric delta")
display(sodium_metric_delta.round(4))
print("Sensitivity tuning results")
display(sensitivity_tuning_results)

sodium_sensitivity_comparison.to_csv(SODIUM_OUTPUT_DIR / "sodium_sensitivity_model_comparison.csv", index=False)
sodium_metric_delta.to_csv(SODIUM_OUTPUT_DIR / "sodium_sensitivity_metric_delta.csv")
sensitivity_tuning_results.to_csv(SODIUM_OUTPUT_DIR / "sodium_sensitivity_tuning_results.csv", index=False)
main_sodium_section_cm.to_csv(SODIUM_OUTPUT_DIR / "main_model_confusion_matrix_for_sodium_comparison.csv")
sodium_sensitivity_cm.to_csv(SODIUM_OUTPUT_DIR / "sodium_sensitivity_confusion_matrix.csv")

Held-out sensitivity comparison. Hyperparameters and thresholds were selected using training OOF predictions only.


,model,feature_count,includes_sodium,best_params,threshold,f1,recall_sensitivity,precision,specificity,balanced_accuracy,accuracy,roc_auc,tn,fp,fn,tp
0,Main CatBoost: BorutaSHAP accepted features only,4,False,"{'depth': 3, 'iterations': 250, 'l2_leaf_reg':...",0.29,0.8539,0.9048,0.8085,0.4375,0.6711,0.7759,0.8095,7,9,4,38
1,Sodium sensitivity CatBoost: accepted features...,5,True,"{'depth': 4, 'iterations': 250, 'l2_leaf_reg':...",0.15,0.8723,0.9762,0.7885,0.3125,0.6443,0.7931,0.7679,5,11,1,41


Main model confusion matrix


,Predicted normal,Predicted abnormal
Actual normal,7,9
Actual abnormal,4,38


Sodium sensitivity model confusion matrix


,Predicted normal,Predicted abnormal
Actual normal,5,11
Actual abnormal,1,41


Sensitivity minus main metric delta


,sodium_minus_main
f1,0.0184
recall_sensitivity,0.0714
precision,-0.0200
specificity,-0.1250
balanced_accuracy,-0.0268
accuracy,0.0172
roc_auc,-0.0417


Sensitivity tuning results


,params,threshold,f1,recall_sensitivity,precision,specificity,balanced_accuracy,accuracy,roc_auc,tn,fp,fn,tp,model
0,"{'depth': 3, 'iterations': 250, 'l2_leaf_reg':...",0.5,0.835913,0.798817,0.876623,0.688525,0.743671,0.769565,0.787370,42,19,34,135,Main CatBoost: BorutaSHAP accepted features only
1,"{'depth': 4, 'iterations': 250, 'l2_leaf_reg':...",0.5,0.835913,0.798817,0.876623,0.688525,0.743671,0.769565,0.787952,42,19,34,135,Main CatBoost: BorutaSHAP accepted features only
2,"{'depth': 4, 'iterations': 150, 'l2_leaf_reg':...",0.5,0.811502,0.751479,0.881944,0.721311,0.736395,0.743478,0.794160,44,17,42,127,Main CatBoost: BorutaSHAP accepted features only
3,"{'depth': 3, 'iterations': 150, 'l2_leaf_reg':...",0.5,0.796117,0.727811,0.878571,0.721311,0.724561,0.726087,0.793772,44,17,46,123,Main CatBoost: BorutaSHAP accepted features only
4,"{'depth': 4, 'iterations': 250, 'l2_leaf_reg':...",0.5,0.832827,0.810651,0.856250,0.622951,0.716801,0.760870,0.783490,38,23,32,137,Sodium sensitivity CatBoost: accepted features...
5,"{'depth': 3, 'iterations': 250, 'l2_leaf_reg':...",0.5,0.817337,0.781065,0.857143,0.639344,0.710205,0.743478,0.788340,39,22,37,132,Sodium sensitivity CatBoost: accepted features...
6,"{'depth': 4, 'iterations': 150, 'l2_leaf_reg':...",0.5,0.803859,0.739645,0.880282,0.721311,0.730478,0.734783,0.796392,44,17,44,125,Sodium sensitivity CatBoost: accepted features...
7,"{'depth': 3, 'iterations': 150, 'l2_leaf_reg':...",0.5,0.801282,0.739645,0.874126,0.704918,0.722282,0.730435,0.799205,43,18,44,125,Sodium sensitivity CatBoost: accepted features...


### 6.5 Conclusion

The following conclusion is generated from the sodium sensitivity comparison and SHAP/dependence outputs above. Sodium remains outside the main accepted-feature CatBoost model unless future validation supports promoting it.

In [27]:
from IPython.display import Markdown

comparison_indexed = sodium_sensitivity_comparison.set_index("model")
main_row = comparison_indexed.loc["Main CatBoost: BorutaSHAP accepted features only"]
sodium_row = comparison_indexed.loc["Sodium sensitivity CatBoost: accepted features + SODIUM"]

f1_delta = float(sodium_row["f1"] - main_row["f1"])
balanced_accuracy_delta = float(sodium_row["balanced_accuracy"] - main_row["balanced_accuracy"])
roc_auc_delta = float(sodium_row["roc_auc"] - main_row["roc_auc"])

sodium_display_values = sodium_shap_display_df["SODIUM"].astype(float).to_numpy()
valid_sodium_signal = np.isfinite(sodium_display_values) & np.isfinite(sodium_shap_values)
if valid_sodium_signal.sum() >= 3:
    sodium_shap_correlation = float(np.corrcoef(sodium_display_values[valid_sodium_signal], sodium_shap_values[valid_sodium_signal])[0, 1])
else:
    sodium_shap_correlation = np.nan

if abs(sodium_shap_correlation) >= 0.30:
    shap_interpretability_statement = (
        f"The sodium SHAP dependence shows a moderate relationship with model contribution "
        f"(correlation {sodium_shap_correlation:.2f}), which may be clinically interpretable with domain review."
    )
elif np.isfinite(sodium_shap_correlation):
    shap_interpretability_statement = (
        f"The sodium SHAP dependence appears weak or non-monotonic "
        f"(correlation {sodium_shap_correlation:.2f}), so clinical interpretability is limited in this run."
    )
else:
    shap_interpretability_statement = "The sodium SHAP dependence pattern could not be summarized reliably from the available held-out rows."

meaningful_gain = (f1_delta > 0.02) and (balanced_accuracy_delta > 0.02)
if meaningful_gain and abs(sodium_shap_correlation) >= 0.30:
    sodium_recommendation = "SODIUM may be discussed as an exploratory sensitivity-model feature, but it should still remain outside the main accepted-feature model pending validation."
elif meaningful_gain:
    sodium_recommendation = "SODIUM improves some held-out metrics in this run, but the SHAP pattern needs clinical review; keep it exploratory/tentative."
else:
    sodium_recommendation = "SODIUM should remain tentative and be discussed only as exploratory unless future validation shows consistent improvement."

sodium_conclusion_md = f"""
**Sodium sensitivity conclusion**

- Adding `SODIUM` changed held-out F1 by `{f1_delta:+.3f}`, balanced accuracy by `{balanced_accuracy_delta:+.3f}`, and ROC-AUC by `{roc_auc_delta:+.3f}` compared with the main BorutaSHAP accepted-feature CatBoost model.
- {shap_interpretability_statement}
- {sodium_recommendation}
- The main final CatBoost model is unchanged and continues to use BorutaSHAP accepted features only; this sodium analysis is exploratory and leakage-safe, with fitting and threshold tuning performed on training data only.
""".strip()

display(Markdown(sodium_conclusion_md))
Path(SODIUM_OUTPUT_DIR / "sodium_conclusion.md").write_text(sodium_conclusion_md, encoding="utf-8")


**Sodium sensitivity conclusion**

- Adding `SODIUM` changed held-out F1 by `+0.018`, balanced accuracy by `-0.027`, and ROC-AUC by `-0.042` compared with the main BorutaSHAP accepted-feature CatBoost model.
- The sodium SHAP dependence shows a moderate relationship with model contribution (correlation -0.50), which may be clinically interpretable with domain review.
- SODIUM should remain tentative and be discussed only as exploratory unless future validation shows consistent improvement.
- The main final CatBoost model is unchanged and continues to use BorutaSHAP accepted features only; this sodium analysis is exploratory and leakage-safe, with fitting and threshold tuning performed on training data only.

716